In [0]:
silver_df = spark.readStream.table("retrailrocket.silver.cleaned_events")

In [0]:
#Real-Time Conversion Funnel
from pyspark.sql import functions as F

conversion = (
    silver_df
    .withWatermark("event_timestamp", "30 minutes")
    .groupBy(
        F.window("event_timestamp", "10 minutes", "5 minutes")
    )
    .agg(
        F.sum(F.when(F.col("event") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event") == "addtocart", 1).otherwise(0)).alias("cart_adds"),
        F.sum(F.when(F.col("event") == "transaction", 1).otherwise(0)).alias("transactions")
    )
    .withColumn(
        "View_to_Cart_Rate",
        F.round(F.try_divide(F.col("cart_adds"), F.col("views")) * 100, 2)
    )
    .withColumn(
        "Cart_to_Purchase_Rate",
        F.round(F.try_divide(F.col("transactions"), F.col("cart_adds")) * 100, 2)
    )
)

In [0]:
conversion.writeStream \
    .outputMode("complete") \
    .option(
        "checkpointLocation",
        "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/gold/conversion_v2") \
    .toTable("retrailrocket.gold.conversion_funnel")

In [0]:
#Finding top trending events
trending = (
    silver_df
    .withWatermark("event_timestamp","30 minutes")
    .groupBy(
        F.window("event_timestamp","15 minutes","5 minutes"),
        "itemid"
    )
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event")=="view",1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event")=="addtocart",1).otherwise(0)).alias("cart_adds")
    )
)

In [0]:
trending.writeStream \
    .outputMode("complete") \
    .option(
        "checkpointLocation",
        "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/gold/trending_v2") \
    .toTable("retrailrocket.gold.trending_items")

In [0]:
#Revenue by Category 
revenue = (
    silver_df
    .filter(F.col("event") == "transaction")
    .withWatermark("event_timestamp", "30 minutes")
    .groupBy(F.window("event_timestamp", "10 minutes"), "categoryid")
    .agg(
        F.sum("price").alias("Revenue"),
        F.count("*").alias("Transactions")
    )
)



In [0]:
revenue.writeStream \
    .outputMode("complete") \
    .option(
        "checkpointLocation",
        "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/gold/revenue_v2") \
    .toTable("retrailrocket.gold.events_revenue")

In [0]:
#Cart Abondment Rate
cart = (
    silver_df
    .filter(F.col("event") == "addtocart")
    .withWatermark("event_timestamp", "1 hour")
)

purchase = (
    silver_df
    .filter(F.col("event") == "transaction")
    .withWatermark("event_timestamp", "1 hour")
)

abandoned = (
    cart.alias("c")
    .join(
        purchase.alias("p"),
        (
            (F.col("c.visitorid") == F.col("p.visitorid")) &
            (F.col("c.itemid") == F.col("p.itemid")) &
            (
                F.col("p.event_timestamp") >= F.col("c.event_timestamp")
            ) &
            (
                F.col("p.event_timestamp") <= 
                F.col("c.event_timestamp") + F.expr("INTERVAL 30 MINUTES")
            )
        ),
        "leftOuter"
    )
    .filter(F.col("p.visitorid").isNull())
    .select(
        "c.*"
    )
)
abandoned.writeStream \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/gold/cart_abandonment_v2"
    ) \
    .toTable("retrailrocket.gold.cart_abandonment")

In [0]:
# Catalog health 
catalog = (
    silver_df
    .withWatermark("event_timestamp", "30 minutes")
    .groupBy(F.window("event_timestamp", "30 minutes"), "itemid")
    .agg(
        F.last("price").alias("CurrentPrice"),
        F.last("available").alias("Availability"),
        F.avg("price").alias("AveragePrice")
    )
)

In [0]:
catalog.writeStream \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/gold/catalog_health_v2"
    ) \
    .toTable("retrailrocket.gold.catalog_health_summary")